In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
housing_raw = pd.read_csv('Housing Price Prediction/data/housing.csv')

**Quick Look at Data**

In [ ]:
housing_raw.head()
housing_raw.info()

In [ ]:
housing_raw['ocean_proximity'].value_counts()
housing_raw.describe()

In [ ]:
housing_raw.hist(bins=50, figsize=(12,8))
plt.show()

**Creating the test Set**

In [ ]:
housing_raw['income_cat'] = pd.cut(housing_raw["median_income"],
                               bins=[0.,1.5,3.0,4.5,6.,np.inf],
                               labels=[1,2,3,4,5])

In [ ]:
cat_counts = housing_raw["income_cat"].value_counts().sort_index()
cat_counts.plot.bar(rot=0,grid=True)
plt.xlabel=("Income Category")
plt.ylabel=("Number of districts")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

strat_train_set, strat_test_set = train_test_split(
    housing_raw,
    test_size=0.2,
    stratify=housing_raw["income_cat"],
    random_state=42
)

In [ ]:
strat_test_set["income_cat"].value_counts() / len(strat_test_set)

In [ ]:
for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

## Explore and Visualize Data to gain Insights

In [ ]:
housing_explore = strat_train_set.copy()

In [ ]:
housing_explore.plot(kind="scatter", x="longitude", y="latitude", grid=True, 
              s=housing_explore["population"]/100, label="population",
              c="median_house_value", cmap = "jet", colorbar = True,
              legend=True, sharex=False, figsize=(10,7))
plt.show()

**Look for Correlations**

In [ ]:
corr_matrix = housing_explore.corr(numeric_only=True)
corr_matrix['median_house_value'].sort_values(ascending=False)
#Alternative: pandas.plotting import scattermatrix

**Experiment with Attribute Combinations**

In [ ]:
housing_explore["rooms_per_house"] = housing_explore["total_rooms"] / housing_explore["households"]
housing_explore["bedrooms_ratio"] = housing_explore["total_bedrooms"] / housing_explore["total_rooms"]
housing_explore["people_per_house"] = housing_explore["population"] / housing_explore["households"]

corr_matrix = housing_explore.corr(numeric_only=True)
corr_matrix["median_house_value"].sort_values(ascending=False)

## **Prepare the Data for ML Algorithms**

In [ ]:
housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

**Clean the data**

In [ ]:
#Fill missing values
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')

In [ ]:
housing_num = housing.select_dtypes(include=[np.number])
imputer.fit(housing_num)

In [ ]:
X = imputer.transform(housing_num)
housing_tr = pd.DataFrame(X, columns=housing_num.columns,
                          index = housing_num.index)

**Handling Text and Categorical Attributes**

In [ ]:
housing_cat = housing[["ocean_proximity"]]
housing_cat.head(8)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

oe = OneHotEncoder(sparse_output=False)
housing_cat_encoded = oe.fit_transform(housing_cat)
#Use instead of ordinal encoder to prevent clustering of nearby values

**Feature Scaling and Transformation**

In [ ]:
from sklearn.preprocessing import StandardScaler

std_scaler = StandardScaler()
housing_num_scaled = std_scaler.fit_transform(housing_num)

In [ ]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

model = TransformedTargetRegressor(
    regressor=LinearRegression(),
    transformer=StandardScaler()
)

model.fit(strat_train_set[["median_income"]], housing_labels)

some_new_data = strat_train_set[["median_income"]].iloc[:5]

predictions = model.predict(some_new_data)

In [ ]:
from sklearn.preprocessing import FunctionTransformer

def add_ratio_features(X):
    X = X.copy()
    X["rooms_per_house"] = X["total_rooms"] / X["households"]
    X["bedrooms_ratio"] = X["total_bedrooms"] / X["total_rooms"]
    X["people_per_house"] = X["population"] / X["households"]
    return X

ratio_adder = FunctionTransformer(add_ratio_features)

**Pipelines**

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

num_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore")
)

In [ ]:
preprocessing = ColumnTransformer([
    ("num", num_pipeline, make_column_selector(dtype_include=np.number)),
    ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
])

full_pipeline = make_pipeline(ratio_adder, preprocessing)

In [ ]:
housing_prepared = full_pipeline.fit_transform(housing)
housing_prepared.shape

## **Select and Train a Model**

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = make_pipeline(full_pipeline, DecisionTreeRegressor(random_state=42))
tree_reg.fit(housing, housing_labels);

In [ ]:
from sklearn.metrics import root_mean_squared_error

housing_predictions = tree_reg.predict(housing)
tree_rmse = root_mean_squared_error(housing_labels,housing_predictions)
tree_rmse

In [ ]:
from sklearn.model_selection import cross_val_score

tree_rmses = -cross_val_score(tree_reg, housing, housing_labels,
                               scoring="neg_root_mean_squared_error", cv=10)

pd.Series(tree_rmses).describe()

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest_reg = make_pipeline(full_pipeline, RandomForestRegressor(random_state=42))
forest_rmses = -cross_val_score(forest_reg, housing, housing_labels,
                                scoring="neg_root_mean_squared_error", cv=10)

pd.Series(forest_rmses).describe()

## **Fine Tune Your Model**

In [ ]:
from sklearn.model_selection import GridSearchCV

forest_reg = make_pipeline(full_pipeline, RandomForestRegressor(random_state=42))

param_grid = [
    {'randomforestregressor__max_features': [4, 6, 8]},
    {'randomforestregressor__max_features': [6, 8, 10],
     'randomforestregressor__n_estimators': [100, 200]},
]

grid_search = GridSearchCV(forest_reg, param_grid, cv=3,
                            scoring='neg_root_mean_squared_error')
grid_search.fit(housing, housing_labels)

In [ ]:
cv_res = pd.DataFrame(grid_search.cv_results_)

cv_res = (
    cv_res.sort_values(by="mean_test_score", ascending=False)
          .assign(RMSE=lambda x: -x["mean_test_score"])
)

# Display only the important columns
display(
    pd.concat(
        [
            cv_res["params"].apply(pd.Series),
            cv_res[["RMSE", "std_test_score", "rank_test_score"]]
        ],
        axis=1,
    ).head(10)
)

## **Finalize Best Model & Evaluate on Test Set**


In [ ]:
import os, json, joblib, tempfile
from pathlib import Path
from sklearn.metrics import root_mean_squared_error

# Best estimator from grid_search
best_model = grid_search.best_estimator_

# Final hold-out evaluation on stratified test set
X_test = strat_test_set.drop("median_house_value", axis=1)
y_test = strat_test_set["median_house_value"].copy()

final_preds = best_model.predict(X_test)
final_rmse = root_mean_squared_error(y_test, final_preds)
final_mae = (y_test - final_preds).abs().mean()

print(f"Final RMSE: {final_rmse:,.2f}")
print(f"Final MAE : {final_mae:,.2f}")


In [ ]:
# Collect baseline CV and test metrics for comparison
baseline = {
    "model": "RandomForest",
    "final_rmse": float(final_rmse),
    "final_mae": float(final_mae),
    "cv_rmse_mean": float(grid_search.best_score_ and -grid_search.best_score_),
}
baseline

## **Small Model Comparison: Ridge vs Random Forest**


In [ ]:
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import cross_val_score

# Ridge via SGD closed-form minimizes squared error with l2 penalty
# Using identity kernel keeps it equivalent to OLS; we use l2 to regularize
ridge_pipeline = make_pipeline(
    FunctionTransformer(add_ratio_features, validate=False),
    ColumnTransformer(
        [
            ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), make_column_selector(dtype_include=np.number)),
            ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), make_column_selector(dtype_include=object)),
        ]
    ),
    SGDRegressor(loss="squared_error", penalty="l2", alpha=0.01, tol=1e-3, random_state=42)
)

# CV for ridge
ridge_rmses = -cross_val_score(ridge_pipeline, housing, housing_labels, scoring="neg_root_mean_squared_error", cv=5)

# Fit on training data for final predictions
ridge_pipeline.fit(housing, housing_labels)
ridge_preds = ridge_pipeline.predict(X_test)
ridge_rmse = root_mean_squared_error(y_test, ridge_preds)
ridge_mae = (y_test - ridge_preds).abs().mean()

cv_ridge_mean = float(ridge_rmses.mean())
baseline2 = {
    "model": "RidgeSGD",
    "final_rmse": float(ridge_rmse),
    "final_mae": float(ridge_mae),
    "cv_rmse_mean": cv_ridge_mean,
}
baseline2


In [ ]:
# Comparison table
import pandas as pd
summary = pd.DataFrame([baseline, baseline2])
summary = summary.assign(RMSE=lambda x: x["final_rmse"], MAE=lambda x: x["final_mae"])
display(summary[["model","cv_rmse_mean","final_rmse","final_mae"]].sort_values("final_rmse"))
# Save best model preference
best = summary.sort_values("final_rmse").iloc[0]
if best["model"] == "RandomForest":
    best_model_to_save = best_model
else:
    best_model_to_save = ridge_pipeline
best_model_to_save


## **Save Artifact**


In [ ]:
import tempfile, json, joblib
from pathlib import Path

art_dir = Path('Housing Price Prediction/artifacts')
art_dir.mkdir(parents=True, exist_ok=True)

model_path = art_dir / 'housing_price_pipeline.joblib'
joblib.dump(best_model_to_save, model_path)

metric_path = art_dir / 'metrics.json'
metrics = {
    'model': best["model"],
    'final_rmse': float(best["final_rmse"]),
    'final_mae': float(best["final_mae"]),
    'cv_rmse_mean': float(best["cv_rmse_mean"]) if not pd.isna(best["cv_rmse_mean"]) else None,
    'artifact': str(model_path),
    'notes': 'Best model selected by final hold-out RMSE on stratified test set.'
}
metric_path.write_text(json.dumps(metrics, indent=2))
print('Saved:', model_path)
print('Metrics:', metric_path)
metrics


## **Predict on New Data**


In [ ]:
new_district = pd.DataFrame([{
    'longitude': -122.23,
    'latitude': 37.88,
    'housing_median_age': 21.0,
    'total_rooms': 1467.0,
    'total_bedrooms': 301.0,
    'population': 787.0,
    'households': 246.0,
    'median_income': 2.5,
    'ocean_proximity': 'NEAR BAY'
}])

price_pred = best_model_to_save.predict(new_district)[0]
print(f'Predicted median house value: \${price_pred:,.2f}')
